### Question 3 Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-3, 4, 250)
mu1, mu2 = 0, 1
pi1, pi2 = 0.4, 0.6

# 1 Local Maximum
var1, var2 = 0.8, 0.8 
gauss1 = (1/(var1 * np.sqrt(2 * np.pi))) * np.exp(-np.power(x-mu1,2)/(2 * np.power(var1, 2)))
gauss2 = (1/(var2 * np.sqrt(2 * np.pi))) * np.exp(-np.power(x-mu2,2)/(2 * np.power(var2, 2)))

gmm1 = pi1* gauss1 + pi2* gauss2

# 2 Local Maxima
var3, var4 = 0.25, 0.25 
gauss3 = (1/(var3 * np.sqrt(2 * np.pi))) * np.exp(-np.power(x-mu1,2)/(2 * np.power(var3, 2)))
gauss4 = (1/(var4 * np.sqrt(2 * np.pi))) * np.exp(-np.power(x-mu2,2)/(2 * np.power(var4, 2)))

gmm2 = pi1* gauss3 + pi1* gauss4

plt.plot(x, gmm1, label='GMM1')
plt.plot(x, gmm2, label='GMM2')
plt.legend()
plt.show()

### Practical Part

We are given 3 datasets containing (x,y) coordinates and the label for each coordinate, our goal is to learn a GMM for each of the sets seperately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

#### GMM Class Implementation

First, we declare a helper function for a multivariate gaussian pdf:
$$f(x_1,...,x_k) = \frac{\exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})\right)}{\sqrt{(2\pi)^k |\boldsymbol{\Sigma}|}} $$

For the class implementation, the intialization inside `fit` was done in the following manner:
* $\mu$ - selecting $k$ random data existing data points in order to break symmetry (discussed in question 2).
* $\Sigma$ - all covariances are initialized to that of a standard normal distribution.
* $\pi$ - uniform along all components $\frac{1}{k}$.
* `animHistory` - aggregated history of the parameters across all steps of the algorithm, for plotting the animations later.

E-step and M-step functions are used to evaluate the $r$ matrix at each step and to maximize the log-likelihood at that step, respectively.

For the sampling function we select the source component for each new synthetic data point according to the learned $\pi$, denoted by `k_choices`. In order to sample from each 'stretched' gaussian/component, we used the cholesky decomposition to find the lower triangular matrix of each covariance matrix $\Sigma = RR^{\top}$ and to apply the affine transformation $x = Rz + \mu$ where $z \sim \mathcal{N}(0, I)$, which we've proven in exercise 1 question 9 to be equivalent to sampling from the 'stretched' gaussian.

We used stablizing epsilons along this implementation to diminish numerical errors caused by programming limits, in theory they should not be applied:
* $\Sigma$ in the gaussian pdf helper function as well as in the sampling function - added $\epsilon \cdot I$.
* Denominator at the E-step and the M-step (in case it is very very small).

In [ ]:
def gaussian_pdf(X, mu, sigma):
        D = X.shape[1]
        epsilon = 1e-6
        sigma_stable = sigma + np.eye(D) * epsilon
        
        diff = X - mu
        inv_sigma = np.linalg.inv(sigma_stable)
        det_sigma = np.linalg.det(sigma_stable)
        
        quad_form = np.sum(np.dot(diff, inv_sigma) * diff, axis=1)
        
        norm_const = 1.0 / (np.power(2 * np.pi, D / 2.0) * np.sqrt(det_sigma))
        return norm_const * np.exp(-0.5 * quad_form)

class GMM:
    def __init__(self, n_components, max_iter=10):
        self.n_components = n_components
        self.max_iter = max_iter

        # Parameter vectors
        self.pi = None
        self.mu = None
        self.sigma = None
        self.animHistory = []

    def fit(self, X):
        n_samples, n_features = X.shape
        
        # Parameter initialization
        self.mu = X[np.random.choice(n_samples, self.n_components, replace=False)]
        self.sigma = np.array([np.eye(n_features) for _ in range(self.n_components)])
        self.pi = np.ones(self.n_components) / self.n_components

        # EM loop
        for _ in range(self.max_iter):
            self.animHistory.append((self.mu.copy(), self.sigma.copy()))
            r = self.e_step(X)          # responsibilities matrix
            self.m_step(X, r)

    def e_step(self, X):
        n_samples = X.shape[0]
        r = np.zeros((n_samples, self.n_components))
        
        for k in range(self.n_components):
            r[:, k] = self.pi[k] * gaussian_pdf(X, self.mu[k], self.sigma[k])
        
        denom = r.sum(axis=1, keepdims=True)
        return r / (denom + 1e-10)

    def m_step(self, X, r):
        n_samples = X.shape[0]
        N_k = r.sum(axis=0)
        
        self.pi = N_k / n_samples
        self.mu = np.dot(r.T, X) / (N_k[:, np.newaxis] + 1e-10)
        
        for k in range(self.n_components):
            diff = X - self.mu[k]
            self.sigma[k] = np.dot(r[:, k] * diff.T, diff) / (N_k[k] + 1e-10)

    def sample(self, n_samples):
        # Normalize weights to ensure they sum to 1 (to prevent numerical errors - in theory this is not relevant)
        weights = self.pi / np.sum(self.pi)
        
        k_choices = np.random.choice(self.n_components, size=n_samples, p=weights)
        n_features = self.mu.shape[1]
        samples = np.zeros((n_samples, n_features))
        
        # Sample generation
        for i, k in enumerate(k_choices):
            # Add small epsilon to diagonal for numerical stability during decomposition
            sigma_stable = self.sigma[k] + np.eye(n_features) * 1e-10
            
            R = np.linalg.cholesky(sigma_stable) 
            z = np.random.standard_normal(n_features)
            
            # Proven in Exerise 1 Q9
            samples[i] = self.mu[k] + np.dot(R, z)
            
        return samples, k_choices

#### Loading the npy file

In [ ]:
full_data = np.load('datasets.npy', allow_pickle=True).item()

#### Displaying the learned Ellipsoids

We expermiented with different amount of iterations, while for `set1` and `set2` only a few had sufficed, `set3` needed a bit more to see any significant convergence/shape in the animation later. We settled for 150 iterations.

In [ ]:
# Helper function for drawing each ellipse
def draw_ellipse(position, covariance, ax, color, alpha):
    U, s, Vt = np.linalg.svd(covariance)
    angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
    width, height = 2 * np.sqrt(s)
    
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(xy=position, width=nsig * width, height=nsig * height, 
                             angle=angle, color=color, alpha=alpha))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = {}

for i, key in enumerate(full_data):
    point_coords, labels = full_data[key] 
    
    n_comp = len(np.unique(labels))
    gmm = GMM(n_components=n_comp, max_iter=150)
    gmm.fit(point_coords)
    models[key] = gmm
    
    # Plotting
    axes[i].scatter(point_coords[:, 0], point_coords[:, 1], c=labels, s=10, alpha=0.5, cmap='viridis')
    for k in range(gmm.n_components):
        draw_ellipse(gmm.mu[k], gmm.sigma[k], ax=axes[i], color='red', alpha=0.2)
    axes[i].set_title(f"Fitted GMM of {key}")

plt.show()

#### Animation

In [ ]:
target_key = 'set3'
X_anim, _ = full_data[target_key] 
X_anim = X_anim[:, :2]

point_coords, labels = full_data[target_key]

# Calculate number of unique classes for n_components
n_comp = len(np.unique(labels))

anim_gmm = GMM(n_components=n_comp, max_iter=150)
anim_gmm.fit(X_anim)

fig_anim, ax_anim = plt.subplots()

def update(frame):
    ax_anim.clear()
    ax_anim.scatter(X_anim[:, 0], X_anim[:, 1], c='gray', s=10, alpha=0.4)
    mu_h, sigma_h = anim_gmm.animHistory[frame]
    for k in range(anim_gmm.n_components):
        draw_ellipse(mu_h[k], sigma_h[k], ax=ax_anim, color='blue', alpha=0.2)
    ax_anim.set_title(f"EM Iteration: {frame}")

animation = FuncAnimation(fig_anim, update, frames=len(anim_gmm.animHistory), interval=200)
plt.close() 
display(HTML(animation.to_jshtml()))

##### All animations (Optional)

In [ ]:
for target_key in full_data:
    X_anim, _ = full_data[target_key] 
    X_anim = X_anim[:, :2]

    point_coords, labels = full_data[target_key]
    
    # Calculate number of unique classes for n_components
    n_comp = len(np.unique(labels))

    anim_gmm = GMM(n_components=n_comp, max_iter=150)
    anim_gmm.fit(X_anim)

    fig_anim, ax_anim = plt.subplots()

    def update(frame):
        ax_anim.clear()
        ax_anim.scatter(X_anim[:, 0], X_anim[:, 1], c='gray', s=10, alpha=0.4)
        mu_h, sigma_h = anim_gmm.animHistory[frame]
        for k in range(anim_gmm.n_components):
            draw_ellipse(mu_h[k], sigma_h[k], ax=ax_anim, color='blue', alpha=0.2)
        ax_anim.set_title(f"EM Iteration: {frame}")

    animation = FuncAnimation(fig_anim, update, frames=len(anim_gmm.animHistory), interval=200)
    plt.close() 
    display(HTML(animation.to_jshtml()))

#### Sampling

In [ ]:
for target_key in full_data:
    gmm_sample = models[target_key]
    new_samples, new_labels = gmm_sample.sample(100)
    
    X, labels = full_data[target_key]

    plt.figure(figsize=(6, 4.5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, alpha=0.2, label='Data')
    plt.scatter(new_samples[:, 0], new_samples[:, 1], c=new_labels, marker='+', s=50,  label='Sampled')

    plt.legend()
    plt.title(f"Sampled Data from GMM {target_key}")
    plt.show()

We can see that the model succeeds in learning the **structure** of the distribtuion sets (only for `set1` and `set2`), we can also see that the labels do not match the original labels necessarily, this is expected because it is an unsupervised model after all. The model did not succeed in capturing the distribution of `set3`, it makes sense because it does not have a structure that can be represented with gaussians very well. In the animation we can see that there are a few small gaussians and one big gaussian, we can assume that it's $\pi$ was quite large in comparison to other components because most sampled points are sccattered randomly.